In [1]:
# CELDA 1 - Conexión
from zeep import Client

wsdl = "https://servayto.madrid.es/MTPAR_WSINFO/InfoParking?wsdl"
client = Client(wsdl=wsdl)

In [2]:
# CELDA 2 - Operaciones disponibles
for service in client.wsdl.services.values():
    for port in service.ports.values():
        for operation in port.binding._operations.values():
            print(operation.name)

GetListParking
GetListFeatures
GetListStreetPoisParking
GetInfoParkingPoisForCoordinate
GetDetailParking


In [3]:
# CELDA 3 - Investigar GetListParking
print(client.service.GetListParking.__doc__)

GetListParking(language: xsd:string) -> GetListParkingResult: ns2:responseListParking


In [4]:
# CELDA 4 - Obtener listado de aparcamientos

resultado = client.service.GetListParking(language="ES")

print(resultado)

XMLParseError: Unexpected element '{http://schemas.datacontract.org/2004/07/InfoParking}ArrayOflstParking', expected '{http://schemas.datacontract.org/2004/07/InfoParking}Data'

In [5]:
from zeep import Client, Settings

settings = Settings(strict=False)

wsdl = "https://servayto.madrid.es/MTPAR_WSINFO/InfoParking?wsdl"

client = Client(
    wsdl=wsdl,
    settings=settings
)

print("Cliente creado")

Cliente creado


In [7]:
resultado = client.service.GetListParking(language="ES")

print(resultado)

{
    'code': 0,
    'message': 'Operación registrada OK.',
    'Data': None,
    '_raw_elements': deque([<Element {http://schemas.datacontract.org/2004/07/InfoParking}ArrayOflstParking at 0x1bea3962500>])
}


In [8]:
from lxml import etree

raw = resultado["_raw_elements"][0]

print(
    etree.tostring(
        raw,
        pretty_print=True,
        encoding="unicode"
    )[:5000]
)

<ns2:ArrayOflstParking xmlns:ns2="http://schemas.datacontract.org/2004/07/InfoParking" xmlns="http://tempuri.org/" xmlns:ns3="http://schemas.microsoft.com/2003/10/Serialization/Arrays" xmlns:soap="http://schemas.xmlsoap.org/soap/envelope/">
  <ns2:lstParking>
    <ns2:address>Calle Corazón de María</ns2:address>
    <ns2:administrativeArea>Madrid</ns2:administrativeArea>
    <ns2:areaCode>28002 </ns2:areaCode>
    <ns2:category>POI Categoria Parking</ns2:category>
    <ns2:country>España</ns2:country>
    <ns2:family>POI Familia</ns2:family>
    <ns2:familyCode>001</ns2:familyCode>
    <ns2:id>3</ns2:id>
    <ns2:latitude>40.438524</ns2:latitude>
    <ns2:longitude>-3.645525</ns2:longitude>
    <ns2:name>Corazón de María II</ns2:name>
    <ns2:nickName>MaríaII</ns2:nickName>
    <ns2:state>Madrid  </ns2:state>
    <ns2:town>Madrid</ns2:town>
    <ns2:type>POI Tipo Parking</ns2:type>
  </ns2:lstParking>
  <ns2:lstParking>
    <ns2:address>Plaza Encuentro</ns2:address>
    <ns2:administr

In [9]:
import pandas as pd

raw = resultado["_raw_elements"][0]

parkings = []

for parking in raw:
    datos = {}

    for campo in parking:
        nombre_campo = campo.tag.split("}")[-1]

        if len(campo) == 0:
            datos[nombre_campo] = campo.text
        else:
            for subcampo in campo:
                nombre_subcampo = subcampo.tag.split("}")[-1]
                datos[f"{nombre_campo}_{nombre_subcampo}"] = subcampo.text

    parkings.append(datos)

df = pd.DataFrame(parkings)

df.head()

,address,administrativeArea,areaCode,category,country,family,familyCode,id,latitude,longitude,name,nickName,state,town,type,lstOccupation_occupation
0,Calle Corazón de María,Madrid,28002,POI Categoria Parking,España,POI Familia,001,3,40.438524,-3.645525,Corazón de María II,MaríaII,Madrid,Madrid,POI Tipo Parking,NaN
1,Plaza Encuentro,Madrid,28030,POI Categoria Parking,España,POI Familia,001,4,40.405464,-3.651354,Encuentro,Encuentro,Madrid,Madrid,POI Tipo Parking,NaN
2,Calle de la Hiedra,Madrid,28036,POI Categoria Parking,España,POI Familia,001,5,40.472181,-3.679160,Nuestra Señora del Recuerdo,Recuerdo,Madrid,Madrid,POI Tipo Parking,NaN
3,Calle Perez de Victoria,Madrid,28023,POI Categoria Parking,España,POI Familia,001,6,40.4568,-3.7832,Corona Boreal,C.Boreal,Madrid,Madrid,POI Tipo Parking,NaN
4,"Avda. de Portugal, s/n. Frente al nº 51",Madrid,28011,POI Categoria Parking,España,POI Familia,001,7,40.415415,-3.727515,Avenida de Portugal,AvPortugal,Madrid,Madrid,POI Tipo Parking,NaN


In [10]:
df.shape

(75, 16)

In [11]:
df.columns.tolist()

['address',
 'administrativeArea',
 'areaCode',
 'category',
 'country',
 'family',
 'familyCode',
 'id',
 'latitude',
 'longitude',
 'name',
 'nickName',
 'state',
 'town',
 'type',
 'lstOccupation_occupation']